In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
WEIGHTS_DIR = "gravnet_truth3b_classifier_faser"   # folder under get_weights_path()
RUN         = 10000
GPU         = "cuda:0"
NUM_EVENTS  = None      # None → load all events per chunk
SAVE_CACHE  = False     # set True to save inference results to disk

CLASS_NAMES      = ["other_mu", "secondary_e", "primary_EM_e"]
NUM_NODE_CLASSES = 3
LABEL_REMAP      = [0, 1, 2, 0]   # pdg_label → truth3b label: other→0, sec_e→1, pEM→2, muon→0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, classification_report, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch_geometric.loader import DataLoader
from tqdm.notebook import tqdm

from analysis.gravnet.model import NeutrinoGravNetNodesFaser
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

# 3-class colour scheme: other_mu=blue, secondary_e=coral, primary_EM_e=green
COLORS    = ["#4477AA", "#EE6677", "#44AA77"]
MARKER_KW = dict(markersize=4, markerfacecolor='white', markeredgewidth=1.2)
C1 = "#353D4C"   # train
C2 = "#E17883"   # val
C3 = "#5691D9"   # AUC

device       = torch.device(GPU if torch.cuda.is_available() else "cpu")
weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / "gravnet_truth3b_classifier" / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

remap_tensor = torch.tensor(LABEL_REMAP)

print(f"Device  : {device}")
print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

__Training curves__

In [ ]:
metrics_path = weights_path / "training_metrics.npz"

if not metrics_path.exists():
    print(f"training_metrics.npz not found — training still in progress.")
else:
    metrics = np.load(metrics_path)
    n_recorded = len(metrics["train_loss"])

    latest_ckpt = weights_path / "latest_checkpoint.pt"
    if latest_ckpt.exists():
        _c          = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
        epoch_end   = _c["epoch"] + 1
        epoch_start = epoch_end - n_recorded + 1
    else:
        epoch_start = 1
        epoch_end   = n_recorded

    epochs  = np.arange(epoch_start, epoch_end + 1)
    best_ep = epoch_start + int(np.argmin(metrics["val_loss"]))

    auc_key   = "val_auc" if "val_auc" in metrics else "val_recall_prim_EM"
    auc_label = "Val AUC-ROC (OvR)" if "val_auc" in metrics else "Val recall (primary_EM_e)"

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.plot(epochs, metrics["train_loss"], marker='o', lw=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_loss"],   marker='s', lw=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
    ax.legend(frameon=False); ax.set_title("Loss")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, metrics["train_acc"], marker='o', lw=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_acc"],   marker='s', lw=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(frameon=False); ax.set_title("Overall accuracy")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(epochs, metrics[auc_key], marker='s', lw=0.8, color=C3, label=auc_label, **MARKER_KW)
    ax.axhline(1/3, color='gray', ls='--', lw=0.8, label="Random (0.33)")
    ax.axvline(best_ep, color=C2, ls=':', lw=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_xlabel("Epoch"); ax.set_ylabel(auc_label)
    ax.legend(frameon=False); ax.set_title("AUC-ROC OvR (val)")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(figures_path / "training_curves.png", dpi=350, bbox_inches='tight')
    plt.show()
    print(f"Epochs plotted : {epoch_start}\u2013{epoch_end}  (recorded: {n_recorded})")
    print(f"Best epoch: {best_ep}  val_loss={metrics['val_loss'][best_ep - epoch_start]:.4f}  "
          f"val_acc={metrics['val_acc'][best_ep - epoch_start]:.4f}  "
          f"{auc_key}={metrics[auc_key][best_ep - epoch_start]:.4f}")
    print("Saved: training_curves.png")

__Load data (held-out validation set)__

In [ ]:
def get_str_from_run(run):
    return ["nue", "num", "nut", "nun"][run % 4]

run_str  = get_str_from_run(RUN)
run_path = torch_path / f"{RUN}/pointnetpp_faser_all_events"

chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
chunk_files = [f for f in chunk_files
               if "_particle_prob" not in f.stem and "_binary_prob" not in f.stem]
print(f"Run {RUN} ({run_str}): {len(chunk_files)} base chunks found")

dataset = []
for chunk_file in chunk_files:
    chunk_data = torch.load(chunk_file, weights_only=False)
    if NUM_EVENTS is not None:
        chunk_data = chunk_data[:NUM_EVENTS]
    dataset.extend(chunk_data)
    print(f"  {chunk_file.name}: {len(chunk_data)} events")

print(f"\nTotal events loaded: {len(dataset)}")

# Apply truth3b label remap before splitting (same as training)
for d in dataset:
    d.label_3b = remap_tensor[d.pdg_label]

# Reproducible val split — must match training (random_state=42, test_size=0.2)
_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set            : {len(val_dataset)} events")

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

__Load model and run inference__

In [ ]:
ckpt = torch.load(weights_path / "best_model.pt", map_location=device, weights_only=False)
cfg  = ckpt.get("model_config", {})

model = NeutrinoGravNetNodesFaser(
    input_dim=1, num_node_classes=NUM_NODE_CLASSES, faser_dim=5,
    n_gravstack=cfg.get("n_gravstack", 3),
    out_channels=cfg.get("out_channels", 16),
    n_feature_transform=cfg.get("n_feature_transform", 16),
    k=cfg.get("k", 12),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])

print(f"Checkpoint epoch : {ckpt['epoch'] + 1}")
print(f"Val loss         : {ckpt.get('val_loss', float('nan')):.4f}")
print(f"Val AUC          : {ckpt.get('val_auc', float('nan')):.4f}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

_ep        = ckpt['epoch']
_vl        = ckpt.get('val_loss', 0.0)
cache_path = figures_path / f"infer_cache_ep{_ep}_vl{_vl:.4f}.npz"

if cache_path.exists():
    print(f"\nLoading inference from cache: {cache_path.name}")
    _c     = np.load(cache_path)
    y_true = _c["y_true"]; y_pred = _c["y_pred"]; y_prob = _c["y_prob"]
    print(f"Loaded {len(y_true):,} nodes.")
else:
    print("\nRunning inference...")
    model.eval()
    all_targets, all_predictions, all_probabilities = [], [], []
    with torch.no_grad():
        for data in tqdm(val_loader, desc="Inference"):
            data = data.to(device)
            if data.x.size(0) == 0:
                continue
            out  = model(data.x, data.pos, data.batch, data.x_faser)
            prob = torch.softmax(out, dim=1)
            pred = out.argmax(dim=1)
            label_3b = remap_tensor.to(device)[data.pdg_label]
            all_targets.append(label_3b.cpu().numpy())
            all_predictions.append(pred.cpu().numpy())
            all_probabilities.append(prob.cpu().numpy())
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_predictions)
    y_prob = np.concatenate(all_probabilities)
    if SAVE_CACHE:
        np.savez_compressed(cache_path, y_true=y_true, y_pred=y_pred, y_prob=y_prob)
        print(f"Saved inference cache: {cache_path.name}")

print(f"\nNodes evaluated : {len(y_true):,}")
print(f"Overall accuracy: {(y_true == y_pred).mean():.4f}")

__Per-class metrics__

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1, 2], zero_division=0
)
print(f"{'Class':<20}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}  {'Support':>12}")
print("-" * 68)
for name, p, r, f, s in zip(CLASS_NAMES, precision, recall, f1, support):
    print(f"{name:<20}  {p:>10.4f}  {r:>10.4f}  {f:>10.4f}  {s:>12,}")

macro_auc = roc_auc_score(
    label_binarize(y_true, classes=[0, 1, 2]), y_prob,
    multi_class="ovr", average="macro"
)
print(f"\nMacro AUC-ROC (OvR): {macro_auc:.4f}")

__Confusion matrices__

In [ ]:
cm           = confusion_matrix(y_true, y_pred)
cm_norm_row  = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm_col  = cm.astype(float) / cm.sum(axis=0, keepdims=True)

def make_annot(cm_raw, cm_norm):
    annot = np.empty_like(cm_raw, dtype=object)
    for i in range(cm_raw.shape[0]):
        for j in range(cm_raw.shape[1]):
            annot[i, j] = f"{cm_norm[i, j]:.2f}\n({cm_raw[i, j]:,})"
    return annot

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(cm_norm_row, annot=make_annot(cm, cm_norm_row), fmt="", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Recall (row fraction)"}, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("Row-normalised (recall)")

sns.heatmap(cm_norm_col, annot=make_annot(cm, cm_norm_col), fmt="", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, cbar_kws={"label": "Precision (col fraction)"}, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title("Column-normalised (precision)")

plt.tight_layout()
plt.savefig(figures_path / "confusion_matrices.png", dpi=350, bbox_inches="tight")
plt.show()

__Per-class ROC curves (one-vs-rest)__

In [ ]:
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])   # (N, 3)

fig, ax = plt.subplots(figsize=(6, 5))
for i, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc_i   = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=1.8, label=f"{name}  (AUC = {roc_auc_i:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-rest ROC curves — truth3b classifier")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(figures_path / "roc_curves.png", dpi=350, bbox_inches='tight')
plt.show()
print(f"Macro AUC-ROC: {macro_auc:.4f}")

__Prediction confidence distribution__

Confidence = $P(\text{predicted class})$, always $\geq 1/3$. Split by correct vs incorrect predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bins = np.linspace(0, 1, 31)

correct_mask = y_pred == y_true
confidence   = y_prob.max(axis=1)

ax.hist(confidence[correct_mask],  bins=bins, density=True, alpha=0.65,
        color="steelblue", label=f"Correct  ({correct_mask.sum():,})", histtype='stepfilled')
ax.hist(confidence[~correct_mask], bins=bins, density=True, alpha=1.0,
        color="#999999", label=f"Incorrect ({(~correct_mask).sum():,})", histtype='step', linewidth=1.5)

ax.set_xlabel("Confidence $P(\\mathrm{predicted\\ class})$")
ax.set_ylabel("Density")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(figures_path / "confidence_distribution.png", dpi=350, bbox_inches="tight")
plt.show()

__Softmax probability distributions per true class__

For each true class, histogram of $P(\text{class})$ split by correct vs incorrect predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bins = np.linspace(0, 1, 31)

for true_idx, (ax, true_name, color) in enumerate(zip(axes, CLASS_NAMES, COLORS)):
    mask         = y_true == true_idx
    prob_col     = y_prob[mask, true_idx]
    correct_mask = y_pred[mask] == true_idx

    ax.hist(prob_col[correct_mask],  bins=bins, alpha=0.65,
            color=color, label=f"Correct  ({correct_mask.sum():,})", density=True)
    ax.hist(prob_col[~correct_mask], bins=bins, alpha=0.50,
            color="#999999", label=f"Incorrect ({(~correct_mask).sum():,})", density=True)
    ax.set_xlabel(f"$P(\\mathtt{{{true_name}}})$", fontsize=10)
    ax.set_ylabel("Density")
    ax.set_title(f"True class: {true_name}")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Predicted softmax probability for each true class", y=1.01)
plt.tight_layout()
plt.savefig(figures_path / "probability_distributions.png", dpi=350, bbox_inches='tight')
plt.show()

__Failure analysis — off-diagonal confusion breakdown__

For each true class, what fraction of its misclassified nodes are predicted as each other class.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for true_idx, (ax, true_name, color) in enumerate(zip(axes, CLASS_NAMES, COLORS)):
    misclassified = (y_true == true_idx) & (y_pred != true_idx)
    n_errors = misclassified.sum()

    if n_errors == 0:
        ax.text(0.5, 0.5, "No errors", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"True: {true_name}")
        continue

    confused_as = y_pred[misclassified]
    counts = np.bincount(confused_as, minlength=3).astype(float)
    counts[true_idx] = 0
    fractions = counts / n_errors

    other_names = [n for i, n in enumerate(CLASS_NAMES) if i != true_idx]
    other_fracs = [fractions[i] for i in range(3) if i != true_idx]
    other_colors = [COLORS[i] for i in range(3) if i != true_idx]

    bars = ax.bar(other_names, other_fracs, color=other_colors, alpha=0.85, edgecolor='white')
    for bar, frac in zip(bars, other_fracs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{frac:.3f}", ha='center', va='bottom', fontsize=8)
    ax.set_ylabel("Fraction of errors")
    ax.set_title(f"True: {true_name}  ({n_errors:,} misclassified)")
    ax.set_ylim(0, 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle("Misclassification breakdown per true class", y=1.02)
plt.tight_layout()
plt.savefig(figures_path / "failure_breakdown.png", dpi=350, bbox_inches='tight')
plt.show()

__Physics characterisation — per-event accuracy vs physics quantities__

Each point is one val event. Black line = binned median.

In [ ]:
# Build per-event records with inference
model.eval()
per_event = []
with torch.no_grad():
    for batch_data in tqdm(val_loader, desc="Scoring val events"):
        batch_data = batch_data.to(device)
        out   = model(batch_data.x, batch_data.pos, batch_data.batch, batch_data.x_faser)
        prob  = torch.softmax(out, dim=1).cpu()
        pred  = out.argmax(dim=1).cpu()
        true  = remap_tensor[batch_data.pdg_label.cpu()]
        pos   = batch_data.pos.cpu()
        batch = batch_data.batch.cpu()
        for g in range(int(batch.max().item()) + 1):
            m = batch == g
            per_event.append({'pred': pred[m].numpy(), 'true': true[m].numpy(),
                              'prob': prob[m].numpy(), 'pos': pos[m].numpy()})

records = []
for idx, ev in enumerate(per_event):
    data  = val_dataset[idx]
    E_nu  = float(data.E_nu)
    E_roe = float(data.E_roe)
    records.append({'idx': idx,
                    'acc': float((ev['pred'] == ev['true']).mean()),
                    'E_nu': E_nu, 'E_roe': E_roe,
                    'inelasticity': E_roe / E_nu if E_nu > 0 else np.nan,
                    'n_nodes': int(data.x.size(0)),
                    'pEM_frac': float((data.pdg_label == 2).float().mean()),
                    'pdg_label': data.pdg_label.numpy(),
                    **ev})

acc_arr  = np.array([r['acc']          for r in records])
Enu_arr  = np.array([r['E_nu']         for r in records])
inel_arr = np.array([r['inelasticity'] for r in records])
nn_arr   = np.array([r['n_nodes']      for r in records])
pEM_arr  = np.array([r['pEM_frac']     for r in records])
print(f"Scored {len(records)} events  |  acc {acc_arr.min():.3f}\u2013{acc_arr.max():.3f}")

def binned_median(x, y, n_bins=15):
    valid  = np.isfinite(x) & np.isfinite(y)
    x, y   = x[valid], y[valid]
    order  = np.argsort(x)
    xs, ys = x[order], y[order]
    chunks = np.array_split(np.arange(len(xs)), n_bins)
    return (np.array([xs[c].mean()     for c in chunks if len(c)]),
            np.array([np.median(ys[c]) for c in chunks if len(c)]))

panels = [
    (Enu_arr,  acc_arr, "$E_\\nu$ (GeV)",   "log"),
    (inel_arr, acc_arr, "Inelasticity $y$", "linear"),
    (nn_arr,   acc_arr, "Nodes per event",  "linear"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (xarr, yarr, xlabel, xscale) in zip(axes, panels):
    ax.scatter(xarr, yarr, c="#4477AA", s=8, alpha=0.30, linewidths=0, rasterized=True)
    cx, cy = binned_median(xarr, yarr)
    ax.plot(cx, cy, color='k', lw=1.5, zorder=5)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Per-event accuracy", fontsize=9)
    ax.set_xscale(xscale)
    ax.set_ylim(0, 1.05)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(figures_path / "acc_vs_physics.png", dpi=350, bbox_inches='tight')
plt.show()

__Spatial distribution of misclassified nodes (pooled across all val events)__

2-D histograms of misclassified node positions in the $xz$ projection. Reveals where in the detector the decision boundary breaks down.

In [ ]:
# Pool positions of misclassified nodes: for each true class, where are the errors?
err_positions = {i: [] for i in range(3)}
for rec in records:
    pos, pred, true = rec['pos'], rec['pred'], rec['true']
    for cls in range(3):
        mask = (true == cls) & (pred != cls)
        if mask.any():
            err_positions[cls].append(pos[mask])

for cls in range(3):
    if err_positions[cls]:
        err_positions[cls] = np.concatenate(err_positions[cls], axis=0)
        print(f"{CLASS_NAMES[cls]} errors: {len(err_positions[cls]):,} nodes")
    else:
        print(f"{CLASS_NAMES[cls]}: no errors")

BINS = 60
cmaps = ["Blues", "Reds", "Greens"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cls in zip(axes, range(3)):
    pos_err = err_positions[cls]
    if len(pos_err) == 0:
        ax.set_visible(False)
        continue
    h, xe, ye, img = ax.hist2d(
        pos_err[:, 2], pos_err[:, 0], bins=BINS, cmap=cmaps[cls], density=True
    )
    plt.colorbar(img, ax=ax, label="Density")
    ax.set_xlabel("$z$ (mm)", fontsize=9)
    ax.set_ylabel("$x$ (mm)", fontsize=9)
    ax.set_title(f"True {CLASS_NAMES[cls]} misclassified", fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle("Spatial distribution of misclassified nodes ($xz$ projection)", y=1.02, fontsize=10)
plt.tight_layout()
plt.savefig(figures_path / "error_spatial_density.png", dpi=350, bbox_inches='tight')
plt.show()

__Secondary electron analysis__

The hardest classification boundary is secondary_e vs primary_EM_e — both are electromagnetic deposits. This cell quantifies the secondary_e confusion rate and the purity of primary_EM_e predictions.

In [ ]:
# Pool all val nodes with original PDG labels for cross-reference
all_pred_node = np.concatenate([r['pred']      for r in records])
all_true_3b   = np.concatenate([r['true']      for r in records])
all_pdg       = np.concatenate([r['pdg_label'] for r in records])

PDG_NAMES = {0: "other/hadronic", 1: "secondary_e", 2: "primary_EM_e", 3: "muon"}

# Rate at which each original PDG class is predicted as primary_EM_e (class 2)
pdg_classes = sorted(np.unique(all_pdg))
pred_pEM_rate = {p: (all_pred_node[(all_pdg == p)] == 2).mean() for p in pdg_classes}
class_counts  = {p: int((all_pdg == p).sum())                   for p in pdg_classes}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: prediction rate per original PDG class
names  = [PDG_NAMES.get(p, str(p)) for p in pdg_classes]
rates  = [pred_pEM_rate[p]         for p in pdg_classes]
colors = [COLORS[2] if p == 2 else (COLORS[1] if p == 1 else "#aaaaaa") for p in pdg_classes]
bars   = axes[0].bar(names, rates, color=colors, alpha=0.85, edgecolor='white')
for bar, rate, p in zip(bars, rates, pdg_classes):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{rate:.3f}\n(n={class_counts[p]:,})",
                 ha='center', va='bottom', fontsize=7.5)
axes[0].set_ylabel("P(predicted primary_EM_e | original PDG)", fontsize=9)
axes[0].set_title("Prediction rate per original particle type", fontsize=9)
axes[0].set_ylim(0, 1.25)
axes[0].axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
axes[0].tick_params(axis='x', labelsize=8, rotation=15)
axes[0].spines[["top", "right"]].set_visible(False)

# Right: purity of primary_EM_e predictions (PDG breakdown)
pred_pEM_mask   = all_pred_node == 2
pdg_in_pred_pEM = all_pdg[pred_pEM_mask]
comp_classes = sorted(np.unique(pdg_in_pred_pEM))
comp_counts  = np.array([(pdg_in_pred_pEM == p).sum() for p in comp_classes], dtype=float)
comp_fracs   = comp_counts / comp_counts.sum()
comp_colors  = [COLORS[2] if p == 2 else (COLORS[1] if p == 1 else "#aaaaaa") for p in comp_classes]
bars2 = axes[1].bar(
    [PDG_NAMES.get(p, str(p)) for p in comp_classes],
    comp_fracs, color=comp_colors, alpha=0.85, edgecolor='white'
)
for bar, frac, cnt in zip(bars2, comp_fracs, comp_counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{frac:.3f}\n({int(cnt):,})",
                 ha='center', va='bottom', fontsize=7.5)
axes[1].set_ylabel("Fraction of predicted primary_EM_e nodes", fontsize=9)
axes[1].set_title(f"Purity of primary_EM_e predictions  (n={pred_pEM_mask.sum():,})", fontsize=9)
axes[1].set_ylim(0, 1.25)
axes[1].tick_params(axis='x', labelsize=8, rotation=15)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(figures_path / "secondary_e_analysis.png", dpi=350, bbox_inches='tight')
plt.show()

if 1 in pred_pEM_rate:
    sec_n     = class_counts[1]
    sec_rate  = pred_pEM_rate[1]
    sec_contam = (pdg_in_pred_pEM == 1).sum() / pred_pEM_mask.sum()
    print(f"secondary_e:  {sec_n:,} nodes,  {sec_rate:.1%} predicted as pEM  "
          f"(contamination in pEM predictions: {sec_contam:.1%})")
if 2 in pred_pEM_rate:
    print(f"primary_EM_e: recall = {pred_pEM_rate[2]:.1%}")

__Event displays — best & worst by node accuracy (2-D)__

Three projections ($xz$, $xy$, $yz$) for the N best and N worst validation events. Correct predictions: small filled circles. Misclassified nodes: crosses — colour = predicted class.

In [ ]:
N_BEST   = 2
N_WORST  = 2
N_RANDOM = 2

VIEWS = [
    ("$xz$", 2, 0, "$z$ (mm)", "$x$ (mm)"),
    ("$xy$", 0, 1, "$x$ (mm)", "$y$ (mm)"),
    ("$yz$", 2, 1, "$z$ (mm)", "$y$ (mm)"),
]

records.sort(key=lambda r: r["acc"])
worst = records[:N_WORST]
best  = records[-N_BEST:]
rng   = np.random.default_rng(seed=42)
pool  = [r for r in records if r['idx'] not in {r2['idx'] for r2 in worst + best}]
rand  = list(rng.choice(pool, size=min(N_RANDOM, len(pool)), replace=False))

selected    = worst + rand + best
group_sizes = [N_WORST, N_RANDOM, N_BEST]
row_labels  = (
    [f"Worst #{i+1}  acc={r['acc']:.3f}" for i, r in enumerate(worst)] +
    [f"Random #{i+1}  acc={r['acc']:.3f}" for i, r in enumerate(rand)]  +
    [f"Best #{i+1}   acc={r['acc']:.3f}" for i, r in enumerate(best)]
)

n_rows = len(selected)
fig, axes = plt.subplots(n_rows, 3, figsize=(13, 3.2 * n_rows),
                         gridspec_kw={"hspace": 0.50, "wspace": 0.35})
if n_rows == 1:
    axes = axes[np.newaxis, :]

sep_after = set()
running = 0
for g in group_sizes[:-1]:
    running += g
    sep_after.add(running - 1)

for row, (rec, row_lbl) in enumerate(zip(selected, row_labels)):
    pos, pred, true = rec["pos"], rec["pred"], rec["true"]
    correct = pred == true

    for col, (view_title, hcol, vcol, xlabel, ylabel) in enumerate(VIEWS):
        ax = axes[row, col]

        for cls_idx, color in enumerate(COLORS):
            mask = (pred == cls_idx) & correct
            if mask.any():
                ax.scatter(pos[mask, hcol], pos[mask, vcol],
                           c=color, s=5, alpha=0.28, linewidths=0, zorder=2, rasterized=True)

        for cls_idx, color in enumerate(COLORS):
            mask = (pred == cls_idx) & ~correct
            if mask.any():
                ax.scatter(pos[mask, hcol], pos[mask, vcol],
                           c=color, s=10, marker='x', linewidths=0.6, zorder=4, rasterized=True)

        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"{row_lbl} · {view_title}" if col == 0 else view_title,
                     fontsize=8, loc="left")

    if row in sep_after:
        for col in range(3):
            axes[row, col].spines["bottom"].set_linestyle((0, (5, 4)))
            axes[row, col].spines["bottom"].set_linewidth(1.0)
            axes[row, col].spines["bottom"].set_color("#888888")
            axes[row, col].spines["bottom"].set_visible(True)

legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=COLORS[0],
                  markersize=6, alpha=0.7, label='predicted other_mu'),
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=COLORS[1],
                  markersize=6, alpha=0.7, label='predicted secondary_e'),
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=COLORS[2],
                  markersize=6, alpha=0.7, label='predicted primary_EM_e'),
    mlines.Line2D([], [], marker='x', color='#555555', markersize=6,
                  linestyle='None', markeredgewidth=0.8, label='misclassified'),
]
fig.legend(handles=legend_handles, loc='upper center',
           bbox_to_anchor=(0.5, 1.01), ncol=4, fontsize=8, frameon=False)
plt.savefig(figures_path / "best_worst_events_2d.png", dpi=350, bbox_inches='tight')
plt.show()